# Lung Cancer CT Detection CAD Framework
### Interactive Exploration & Demo Notebook
This notebook walks you through:
1. Verifying your GPU (NVIDIA GeForce RTX 3050)
2. Testing the patient-stratified data loader (Zero Data Leakage)
3. Visualizing the CBAM Attention Module
4. Generating Grad-CAM Heatmaps

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add project root to Python path
sys.path.insert(0, os.path.abspath('..'))

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Using GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.1f} MB')
else:
    print('Running on CPU')

## 1. Verify Model Architecture (EfficientNet-B1 + CBAM)

In [ ]:
from src.models import build_efficientnet_cbam

model = build_efficientnet_cbam(num_classes=3, pretrained=False)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Trainable Parameters: {total_params:,}')

# Test dummy forward pass
dummy_ct_slice = torch.randn(1, 3, 240, 240)
output = model(dummy_ct_slice)
print('Output Logits Shape:', output.shape)
print('Predicted Class Probabilities:', torch.softmax(output, dim=1).detach().numpy())

## 2. Test Grad-CAM Explainability & Attention Visualization

In [ ]:
from src.explain import GradCAM

target_layer = model.features[-1]
grad_cam = GradCAM(model, target_layer)
heatmap = grad_cam.generate_heatmap(dummy_ct_slice, class_idx=2)

plt.figure(figsize=(6, 5))
plt.imshow(heatmap, cmap='jet')
plt.colorbar(label='Attention Weight')
plt.title('Grad-CAM Nodule Saliency Map')
plt.show()